<a href="https://colab.research.google.com/github/dilliprasad570-cpu/LLM/blob/main/Copy_of_Building_LLM_Powered_Apps_with_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🖼️ NIAT Demo: AI Prompt Engineering + Image Generation & Editing

**Run this notebook on Google Colab with a GPU runtime (or via VS Code + Colab extension).**

Pipeline:
1. **Qwen 2.5-0.5B** — rewrites your simple prompt into a detailed, creative one
2. **Stable Diffusion v1.5** — generates/edits images from the engineered prompt
3. **Gradio UI** — provides a web interface with a shareable public link



## Verify GPU Runtime

Go to **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f" GPU: {gpu_name}")
else:
    print(" No GPU! Go to Runtime → Change runtime type → T4 GPU")

---
## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers diffusers accelerate gradio sentencepiece
print("Dependencies installed.")

---
## Step 2: Load Qwen 2.5-0.5B for Prompt Engineering

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading {QWEN_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL, torch_dtype=torch.float16, device_map="auto"
)
print(f"Qwen loaded on {qwen_model.device}")

In [ ]:
def _qwen_generate(messages, max_new_tokens=120, temperature=0.7):
    """Internal: run Qwen generation with chat template."""
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(qwen_model.device)
    output = qwen_model.generate(
        **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
        do_sample=True, pad_token_id=tokenizer.eos_token_id
    )
    generated_tokens = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


def engineer_generation_prompt(user_prompt, max_new_tokens=120, temperature=0.7):
    """Engineer a detailed, creative prompt for image generation."""
    messages = [
        {"role": "system", "content": (
            "You are an expert prompt engineer for AI image generation. "
            "Rewrite the following user prompt to be more detailed, vivid, and creative — "
            "specify style, lighting, composition, and relevant visual details. "
            "Output only the improved prompt.Make the prompt concise but detailed. Put the most important visual elements first."
        )},
        {"role": "user", "content": user_prompt}
    ]
    return _qwen_generate(messages, max_new_tokens, temperature)


def engineer_editing_prompt(user_prompt, max_new_tokens=80, temperature=0.7):
    """Engineer a clear, specific prompt for image editing."""
    messages = [
        {"role": "system", "content": (
            "You are an expert prompt engineer for AI image editing. "
            "Rewrite the following user prompt to be clear and specific about what changes "
            "to make to the existing image. Focus on specific edits needed: style changes, "
            "object modifications, color adjustments. Keep it concise. "
            "Output only the improved prompt."
        )},
        {"role": "user", "content": user_prompt}
    ]
    return _qwen_generate(messages, max_new_tokens, temperature)

In [ ]:
# Quick test
test = "a cat sitting on a table"
print(f"Original:    {test}")
print(f"Engineered:  {engineer_generation_prompt(test)}")

---
## Step 3: Load Stable Diffusion (Shared Weights — Single Download)

In [ ]:
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline

SD_MODEL = "runwayml/stable-diffusion-v1-5"

# Load ONCE — download fp16 variant directly (half the download size)
print("Loading Stable Diffusion...")
txt2img_pipe = StableDiffusionPipeline.from_pretrained(
    SD_MODEL, torch_dtype=torch.float16, variant="fp16"
)
txt2img_pipe = txt2img_pipe.to("cuda")
print("Text-to-Image pipeline loaded.")

# Reuse SAME weights for img2img — no second download!
img2img_pipe = StableDiffusionImg2ImgPipeline(**txt2img_pipe.components)
print("Image Editing pipeline loaded (shared weights).")

In [ ]:
def generate_image(prompt, num_inference_steps=30, guidance_scale=7.5):
    """Generate an image from a prompt. Returns a PIL Image."""
    return txt2img_pipe(prompt, num_inference_steps=num_inference_steps, guidance_scale=guidance_scale).images[0]


def edit_image(init_image, prompt, strength=0.7, guidance_scale=8):
    """Edit an image using Img2Img. Returns a PIL Image."""
    return img2img_pipe(prompt=prompt, image=init_image, strength=strength, guidance_scale=guidance_scale).images[0]

---
## Step 4: Quick Test (prompt + Image)

In [ ]:
from IPython.display import display

user_prompt = "a big cat in a jungle"
print(f"User prompt: {user_prompt}")

engineered = engineer_generation_prompt(user_prompt)
print(f"Engineered:  {engineered}")

print("Generating image...")
img = generate_image(engineered)
display(img)
print("Done!")

---
## Step 5: Launch Gradio UI

Launches the full web interface. A **public shareable link** will be printed.

Open that link in your browser.

In [ ]:
import gradio as gr
from PIL import Image


def generate_image_workflow(prompt):
    try:
        engineered = engineer_generation_prompt(prompt)
        img = generate_image(engineered)
        return engineered, img, img
    except Exception as e:
        return f"ERROR: {str(e)}", None, None


def upload_image_workflow(image):
    return image, image


def edit_image_workflow(image, instruction):
    if image is None:
        return "No image to edit!", None, None
    try:
        engineered = engineer_editing_prompt(instruction)
        edited = edit_image(image, engineered)
        return engineered, edited, edited
    except Exception as e:
        return f"ERROR: {str(e)}", None, None


with gr.Blocks(
    title="NIAT Demo: Prompt-to-Image & Editing",
    css="""
    @import url('https://fonts.googleapis.com/css2?family=Architects+Daughter&display=swap');
    html, body, .gr-block, .gr-box, .gr-button, .gr-textbox, .gr-image, .gr-markdown {
        font-family: 'Architects Daughter', 'Comic Sans MS', cursive, sans-serif !important;
    }
    .sketch-row { display: flex; flex-direction: row; align-items: stretch; gap: 32px; width: 100%; }
    .sketch-col { flex: 1 1 0; display: flex; flex-direction: column; }
    .sketch-box {
        border: 2.5px dashed #2176ae; border-radius: 18px; background: #e3f6fd;
        box-shadow: 2px 2px 0 0 #2176ae, 4px 4px 0 0 #38b6ff;
        padding: 8px; margin-bottom: 10px;
    }
    .sketch-header { font-size:1.3em; font-weight:bold; margin-bottom:0.5em; color:#2176ae; text-shadow: 1px 1px 0 #38b6ff; }
    .gr-button { font-size:1.1em; border-radius:12px; border:2px solid #2176ae; background:#38b6ff; color:#fff; font-family: 'Architects Daughter', cursive !important; }
    .gr-button:hover { background:#2176ae; }
    .sketch-image { border:2.5px dashed #2176ae; border-radius:14px; background:#e3f6fd; box-shadow: 2px 2px 0 0 #2176ae; margin-bottom: 16px; min-height: 160px; height: 160px; }
    .big-prompt-box textarea { min-height: 170px !important; font-size: 1.35em !important; }
    """
) as demo:
    gr.Markdown("""
    # NIAT Demo: Prompt-to-Image & Image Editing
    <span style='font-size:1.1em;'>Running on Colab GPU — Qwen 2.5-0.5B + Stable Diffusion v1.5</span>
    """)

    with gr.Row(elem_classes=["sketch-row"]):
        with gr.Column(scale=1, elem_classes=["sketch-col"]):
            gr.Markdown("<div class='sketch-header'>1. Generate Image from Prompt</div>")
            prompt = gr.Textbox(label="Prompt", placeholder="A cat riding a bicycle in Paris", lines=7, elem_classes=["sketch-box", "big-prompt-box"])
            gen_btn = gr.Button("Generate Image", variant="primary")
            engineered_prompt = gr.Textbox(label="Engineered Prompt (LLM output)", interactive=False, visible=True, elem_classes=["sketch-box"])
            gr.Markdown("<div class='sketch-header'>2. Upload Image to Edit</div>")
            upload = gr.Image(label="Upload Image", type="pil", height=80, elem_classes=["sketch-image"])
            upload_btn = gr.Button("Upload Image", variant="primary")

        with gr.Column(scale=1, elem_classes=["sketch-col"]):
            gr.Markdown("<div class='sketch-header'>3. View & Edit Image</div>")
            orig_image = gr.Image(label="Original Image", type="pil", interactive=False, height=180, elem_classes=["sketch-image"])
            edit_instruction = gr.Textbox(label="Edit Instructions", placeholder="Make the cat wear sunglasses", lines=2, elem_classes=["sketch-box"])
            edit_btn = gr.Button("Edit Image", variant="primary")
            engineered_edit = gr.Textbox(label="Engineered Edit Prompt (LLM output)", interactive=False, visible=True, elem_classes=["sketch-box"])
            edit_image_out = gr.Image(label="Edited Image", type="pil", interactive=False, height=180, elem_classes=["sketch-image"])

    state_image = gr.State()

    gen_btn.click(generate_image_workflow, inputs=prompt, outputs=[engineered_prompt, orig_image, state_image])
    upload_btn.click(upload_image_workflow, inputs=upload, outputs=[orig_image, state_image])
    edit_btn.click(edit_image_workflow, inputs=[state_image, edit_instruction], outputs=[engineered_edit, edit_image_out, orig_image])

# Launch — public link only, no inline rendering
demo.launch(share=True, inline=False, quiet=True)

In [ ]:
try:
    print(f"Your Gradio interface is running at: {demo.share_url}")
except NameError:
    print("The 'demo' object is not defined. Please ensure you have run the Gradio UI cell above first.")
except Exception as e:
    print(f"Could not retrieve URL: {e}")

In [ ]:
test_prompt = "a cat sitting on a chair"
print(engineer_generation_prompt(test_prompt))

---
##

After running the cell above, Gradio will print a **public URL** like:
```
Running on public URL: https://xxxxx.gradio.live
```

Open that link in your browser and run Inference.

